# 宁德时代 (CATL) A+H 双股数据分析报告

> 数据来源：Tushare Pro · 分析日期：2026-06-30

本 Notebook 将展示：
1. 加载宁德时代 A股 (300750.SZ) 和 H股 (03750.HK) 的近一年日线行情数据
2. 数据清洗与基本统计
3. K线图与成交量图绘制
4. A/H 双股对比分析


In [ ]:
# ========== 导入依赖 ==========
import json
import pandas as pd
import numpy as np
import matplotlib
try:
    matplotlib.use('Agg')
except Exception:
    pass
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')

# 中文显示设置
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 依赖加载完成")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy:  {np.__version__}")
print(f"   Matplotlib: {matplotlib.__version__}")


## 1. 加载数据

数据已通过 Tushare Pro MCP 接口获取，存储在本地 JSON 文件中。


In [ ]:
# ====== 加载数据 ======
import os
# 自动检测数据文件目录
DATA_DIR = os.getcwd()
# 备用路径：如果上面无效则用绝对路径
if not os.path.exists(os.path.join(DATA_DIR, "catl_data.json")):
    backup = r"D:\SJM\14、大二下\光华BA量化暑期营\在线实习"
    if os.path.exists(os.path.join(backup, "catl_data.json")):
        DATA_DIR = backup
print(f"[INFO] 数据目录: {DATA_DIR}")

# 加载 A 股数据
with open(os.path.join(DATA_DIR, "catl_data.json"), "r", encoding="utf-8") as f:
    a_data = json.load(f)
df_a = pd.DataFrame(a_data)

# 加载 H 股数据
with open(os.path.join(DATA_DIR, "catl_hk_data.json"), "r", encoding="utf-8") as f:
    h_data = json.load(f)
df_h = pd.DataFrame(h_data)

# 日期排序
df_a = df_a.sort_values("trade_date").reset_index(drop=True)
df_h = df_h.sort_values("trade_date").reset_index(drop=True)

# 日期列转换为 datetime 类型
df_a["trade_date"] = pd.to_datetime(df_a["trade_date"])
df_h["trade_date"] = pd.to_datetime(df_h["trade_date"])

print(f"[OK] A股 300750.SZ：{len(df_a)} 行, {df_a["trade_date"].min().date()} ~ {df_a["trade_date"].max().date()}")
print(f"[OK] H股 03750.HK：{len(df_h)} 行, {df_h["trade_date"].min().date()} ~ {df_h["trade_date"].max().date()}")
print("   H股上市时间: 2025-05-20 (IPO ¥263 HKD)")

In [ ]:
# ========== 数据预览 ==========
print("="*60)
print("A股 300750.SZ 前5行:")
print("="*60)
cols_show = ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'vol', 'pct_chg']
print(df_a[cols_show].head().to_string(index=False))

print("\n" + "="*60)
print("A股 300750.SZ 后5行:")
print("="*60)
print(df_a[cols_show].tail().to_string(index=False))

print("\n" + "="*60)
print("H股 03750.HK 前5行:")
print("="*60)
print(df_h[cols_show].head().to_string(index=False))

print("\n" + "="*60)
print("H股 03750.HK 后5行:")
print("="*60)
print(df_h[cols_show].tail().to_string(index=False))


## 2. 基本统计指标

In [ ]:
# ========== 基本统计 ==========
def print_stats(df, name):
    """打印基本统计指标"""
    first = df.iloc[0]
    last = df.iloc[-1]
    total_ret = (last['close'] - first['close']) / first['close'] * 100
    max_high = df['high'].max()
    min_low = df['low'].min()
    max_date = df.loc[df['high'].idxmax(), 'trade_date'].strftime('%Y-%m-%d')
    min_date = df.loc[df['low'].idxmin(), 'trade_date'].strftime('%Y-%m-%d')
    avg_vol = df['vol'].mean()
    # 波动率
    daily_vol = df['pct_chg'].std()
    annual_vol = daily_vol * np.sqrt(252)
    # 涨跌天数
    up_days = (df['pct_chg'] > 0).sum()
    down_days = (df['pct_chg'] < 0).sum()
    
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  起始价: ¥{first['close']:.2f} ({first['trade_date'].strftime('%Y-%m-%d')})")
    print(f"  最新价: ¥{last['close']:.2f} ({last['trade_date'].strftime('%Y-%m-%d')})")
    print(f"  区间涨跌幅: {total_ret:+.2f}%")
    print(f"  最高价: ¥{max_high:.2f} ({max_date})")
    print(f"  最低价: ¥{min_low:.2f} ({min_date})")
    print(f"  涨/跌天数: {up_days} / {down_days}")
    print(f"  日均成交量: {avg_vol:.0f}")
    print(f"  日波动率: {daily_vol:.2f}%")
    print(f"  年化波动率: {annual_vol:.1f}%")

print_stats(df_a, "A股 宁德时代 300750.SZ")
print_stats(df_h, "H股 宁德时代 03750.HK")

# A/H 溢价
rate = 0.93  # 1 HKD ≈ 0.93 CNY
a_price = df_a.iloc[-1]['close']
h_price = df_h.iloc[-1]['close']
a_in_hkd = a_price / rate
premium = (h_price - a_in_hkd) / a_in_hkd * 100
print(f"\n{'='*50}")
print(f"  A/H 溢价分析")
print(f"{'='*50}")
print(f"  A股收盘价: ¥{a_price:.2f} ≈ HK${a_in_hkd:.2f}")
print(f"  H股收盘价: HK${h_price:.2f}")
print(f"  溢价率: {premium:+.2f}%")
if premium > 0:
    print(f"  → H股相对A股溢价 {premium:.2f}%")
else:
    print(f"  → H股相对A股折价 {abs(premium):.2f}%")


## 3. K线图 + 成交量图

按照中国股市惯例：**涨 = 红色 🔴**，**跌 = 绿色 🟢**

In [ ]:
# ========== K线图绘制函数 ==========
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

def plot_candlestick_volume(df, title, price_label='¥', vol_label='手', save_path=None):
    """绘制K线图 + 成交量柱状图"""
    dates = mdates.date2num(df['trade_date'].values)
    opens = df['open'].values
    highs = df['high'].values
    lows = df['low'].values
    closes = df['close'].values
    vols = df['vol'].values
    pcts = df['pct_chg'].values
    
    # 颜色：涨红跌绿
    up = closes >= opens
    colors = ['#ff4d4f' if u else '#52c41a' for u in up]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10), 
                                     gridspec_kw={'height_ratios': [3, 1]},
                                     sharex=True)
    fig.suptitle(title, fontsize=18, fontweight='bold', y=0.98)
    
    # ---- K线图 ----
    for i in range(len(df)):
        x = dates[i]
        o, h, l, c = opens[i], highs[i], lows[i], closes[i]
        color = colors[i]
        # 影线
        ax1.plot([x, x], [l, h], color=color, linewidth=0.8)
        # 实体
        if up[i]:
            rect = Rectangle((x, o), 0.6, c - o, facecolor=color, edgecolor=color, linewidth=0.5)
        else:
            rect = Rectangle((x, c), 0.6, o - c, facecolor=color, edgecolor=color, linewidth=0.5)
        ax1.add_patch(rect)
    
    # 均线
    if len(df) >= 5:
        df['MA5'] = df['close'].rolling(5).mean()
        ax1.plot(dates, df['MA5'].values, color='#1677ff', linewidth=1.5, label='MA5', alpha=0.8)
    if len(df) >= 10:
        df['MA10'] = df['close'].rolling(10).mean()
        ax1.plot(dates, df['MA10'].values, color='#fa8c16', linewidth=1.5, label='MA10', alpha=0.8)
    if len(df) >= 20:
        df['MA20'] = df['close'].rolling(20).mean()
        ax1.plot(dates, df['MA20'].values, color='#722ed1', linewidth=1.5, label='MA20', alpha=0.8)
    if len(df) >= 60:
        df['MA60'] = df['close'].rolling(60).mean()
        ax1.plot(dates, df['MA60'].values, color='#13c2c2', linewidth=1.5, label='MA60', alpha=0.8)
    
    ax1.set_ylabel(f'价格 ({price_label})', fontsize=12)
    ax1.legend(loc='upper left', fontsize=10, framealpha=0.8)
    ax1.grid(True, alpha=0.3)
    ax1.set_title('K线 (Candlestick)', fontsize=13, loc='left')
    
    # ---- 成交量图 ----
    ax2.bar(dates, vols, width=0.6, color=colors, alpha=0.7)
    ax2.set_ylabel(f'成交量 ({vol_label})', fontsize=12)
    ax2.set_xlabel('交易日期', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.set_title('成交量 (Volume)', fontsize=13, loc='left')
    
    # 日期格式
    ax2.xaxis.set_major_locator(mdates.MonthLocator())
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✅ 图片已保存: {save_path}")
    plt.show()

print("✅ K线图绘制函数已定义")


In [ ]:
# ========== 绘制A股K线图 ==========
plot_candlestick_volume(
    df_a.copy(),
    title='宁德时代 A股 300750.SZ · 近一年日线行情',
    price_label='¥ CNY',
    vol_label='手',
    save_path='catl_ashare_kline.png'
)


In [ ]:
# ========== 绘制H股K线图 ==========
plot_candlestick_volume(
    df_h.copy(),
    title='宁德时代 H股 03750.HK · 近一年日线行情',
    price_label='HK$ HKD',
    vol_label='股',
    save_path='catl_hshare_kline.png'
)


## 4. A/H 双股对比分析

In [ ]:
# ========== A/H 价格走势对比图 ==========
# 按日期合并
merged = pd.merge(df_a[['trade_date', 'close', 'vol']], 
                  df_h[['trade_date', 'close', 'vol']],
                  on='trade_date', suffixes=('_a', '_h'))
merged.columns = ['trade_date', 'close_a', 'vol_a', 'close_h', 'vol_h']

# 归一化（基准日=100）
base_a = merged['close_a'].iloc[0]
base_h = merged['close_h'].iloc[0]
merged['norm_a'] = merged['close_a'] / base_a * 100
merged['norm_h'] = merged['close_h'] / base_h * 100

# 汇率折算 A 股到 HKD
rate = 0.93
merged['close_a_hkd'] = merged['close_a'] / rate
merged['premium'] = (merged['close_h'] - merged['close_a_hkd']) / merged['close_a_hkd'] * 100

print(f"共同交易日数: {len(merged)}")
print(f"A股起始: ¥{base_a:.2f}  →  结束: ¥{merged['close_a'].iloc[-1]:.2f}")
print(f"H股起始: HK${base_h:.2f} →  结束: HK${merged['close_h'].iloc[-1]:.2f}")
print(f"最新溢价率: {merged['premium'].iloc[-1]:+.2f}%")


In [ ]:
# ========== 走势叠加对比图 ==========
fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)
fig.suptitle('宁德时代 A/H 双股对比分析', fontsize=18, fontweight='bold', y=0.98)

dates = merged['trade_date'].values

# 图1: 归一化走势
ax = axes[0]
ax.plot(dates, merged['norm_a'], color='#ff4d4f', linewidth=2, label='A股 300750.SZ', alpha=0.9)
ax.plot(dates, merged['norm_h'], color='#52c41a', linewidth=2, label='H股 03750.HK', alpha=0.9)
ax.axhline(y=100, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_ylabel('归一化价格 (基准=100)', fontsize=12)
ax.legend(loc='upper left', fontsize=11, framealpha=0.8)
ax.grid(True, alpha=0.3)
ax.set_title('价格走势对比 (归一化)', fontsize=13, loc='left')

# 图2: 实际价格
ax = axes[1]
ax.plot(dates, merged['close_a'], color='#ff4d4f', linewidth=1.8, label='A股 (¥)', alpha=0.9)
ax.plot(dates, merged['close_h'], color='#52c41a', linewidth=1.8, label='H股 (HK$)', alpha=0.9)
ax.set_ylabel('价格', fontsize=12)
ax.legend(loc='upper left', fontsize=11, framealpha=0.8)
ax.grid(True, alpha=0.3)
ax.set_title('实际价格对比', fontsize=13, loc='left')

# 图3: 溢价率
ax = axes[2]
ax.fill_between(dates, 0, merged['premium'],
                 where=merged['premium'] >= 0,
                 color='#ff4d4f', alpha=0.3, label='H股溢价')
ax.fill_between(dates, 0, merged['premium'],
                 where=merged['premium'] < 0,
                 color='#52c41a', alpha=0.3, label='H股折价')
ax.plot(dates, merged['premium'], color='#333', linewidth=1.5)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8)
ax.set_ylabel('溢价率 (%)', fontsize=12)
ax.set_xlabel('交易日期', fontsize=12)
ax.legend(loc='upper left', fontsize=11, framealpha=0.8)
ax.grid(True, alpha=0.3)
ax.set_title('A/H 溢价率 (正=H股贵, 负=H股便宜)', fontsize=13, loc='left')

# 日期格式
axes[2].xaxis.set_major_locator(mdates.MonthLocator())
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('catl_ah_comparison.png', dpi=150, bbox_inches='tight')
print("✅ 对比图已保存: catl_ah_comparison.png")
plt.show()


In [ ]:
# ========== 收益率相关性分析 ==========
merged['ret_a'] = merged['close_a'].pct_change() * 100
merged['ret_h'] = merged['close_h'].pct_change() * 100
merged = merged.dropna()

corr = merged['ret_a'].corr(merged['ret_h'])
print(f"A/H 日收益率相关系数: {corr:.4f}")
print()

# 统计描述
print("A股日收益率统计:")
print(merged['ret_a'].describe().apply(lambda x: f'{x:.2f}%'))
print()
print("H股日收益率统计:")
print(merged['ret_h'].describe().apply(lambda x: f'{x:.2f}%'))

# 散点图
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(merged['ret_a'], merged['ret_h'], alpha=0.5, s=15, c='#1677ff')
# 回归线
z = np.polyfit(merged['ret_a'], merged['ret_h'], 1)
p = np.poly1d(z)
x_line = np.linspace(merged['ret_a'].min(), merged['ret_a'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'拟合线 (斜率={z[0]:.3f})')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
ax.set_xlabel('A股日收益率 (%)', fontsize=12)
ax.set_ylabel('H股日收益率 (%)', fontsize=12)
ax.set_title(f'A/H 日收益率散点图 (相关系数: {corr:.4f})', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('catl_ah_correlation.png', dpi=150, bbox_inches='tight')
print("✅ 相关性图已保存: catl_ah_correlation.png")
plt.show()


In [ ]:
# ========== 成交量对比分析 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# A股成交量分布
ax = axes[0]
ax.hist(merged['vol_a'] / 1e4, bins=30, color='#ff4d4f', alpha=0.7, edgecolor='white')
ax.axvline(x=merged['vol_a'].mean() / 1e4, color='#ff4d4f', linestyle='--', 
           linewidth=2, label=f'均值: {merged["vol_a"].mean()/1e4:.0f}万手')
ax.set_xlabel('成交量 (万手)', fontsize=11)
ax.set_ylabel('频次', fontsize=11)
ax.set_title('A股 成交量分布', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# H股成交量分布
ax = axes[1]
ax.hist(merged['vol_h'] / 1e4, bins=30, color='#52c41a', alpha=0.7, edgecolor='white')
ax.axvline(x=merged['vol_h'].mean() / 1e4, color='#52c41a', linestyle='--',
           linewidth=2, label=f'均值: {merged["vol_h"].mean()/1e4:.0f}万股')
ax.set_xlabel('成交量 (万股)', fontsize=11)
ax.set_ylabel('频次', fontsize=11)
ax.set_title('H股 成交量分布', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('catl_ah_volume_dist.png', dpi=150, bbox_inches='tight')
print("✅ 成交量分布图已保存")
plt.show()


## 5. 分析总结

### 关键发现

| 指标 | A股 300750.SZ | H股 03750.HK |
|------|:------------:|:------------:|
| 代码 | 300750.SZ | 03750.HK |
| 交易所 | 深交所创业板 | 港交所主板 |
| 货币 | 人民币 (¥) | 港币 (HK$) |
| IPO时间 | 2018年 | 2025-05-20 |
| A/H 溢价率 | — | 动态变化 |

### 技术分析要点

1. **价格走势**：A股和H股走势高度同步，反映公司基本面驱动为主
2. **溢价率**：H股相对A股存在一定溢价/折价，受两地市场情绪和流动性影响
3. **波动率**：H股作为次新股波动通常大于A股
4. **成交量**：A股流动性显著高于H股
5. **投资建议**：仅作技术展示，不构成投资建议

> ⚠️ **免责声明**：本 Notebook 仅用于学习和展示目的，不构成任何投资建议。


## 附：交互式图表

更丰富的交互式双股对比分析面板请参见 [catl_ah_compare.html](./catl_ah_compare.html)（使用 ECharts 绘制，支持缩放、悬停查看详情等交互功能）。
